# Azure Supplement — Memory & Compute Budgets

> **Source notes:** `memory-budgets.md`  
> **Azure services:** Azure ML compute clusters, GPU VM SKUs (NC-series, ND-series, NV-series)

This supplement demonstrates VRAM estimation and memory profiling on Azure GPU instances.  
It answers: **"Which Azure GPU SKU do I need for my model?"**

**What this supplement covers:**

| Step | Description | Azure Service |
|------|-------------|---------------|
| 1 | Azure credentials setup (empty strings — fill before running) | Azure Identity |
| 2 | Azure SDK setup | `azure-ai-ml`, `azure-identity` |
| 3 | Azure GPU SKU comparison (VRAM, cost, performance) | Azure Pricing API |
| 4 | VRAM estimation for Llama-3-8B on Azure | — |
| 5 | Memory profiling on Azure ML compute cluster | Azure ML |
| 6 | Batch size vs. VRAM tradeoff on Azure | — |
| 7 | Choosing the right Azure GPU for inference vs. training | — |

**Prerequisites:**
- Azure subscription with quota for GPU VMs (NC, ND, or NV series)
- Azure ML workspace created (can use free tier for profiling, but GPU quota required for actual runs)
- Azure CLI installed and authenticated: `az login`

## Step 0 — Azure Credentials Setup

**⚠️ IMPORTANT: Fill in your Azure credentials before running.**

These cells are left empty to pass Git hooks. Replace `""` with your actual values:

```python
AZURE_SUBSCRIPTION_ID = "your-subscription-id-here"
AZURE_RESOURCE_GROUP = "your-resource-group"
AZURE_ML_WORKSPACE = "your-ml-workspace-name"
AZURE_REGION = "eastus"  # or your preferred region
```

**Tip:** Use Azure Key Vault for production. For learning, you can use environment variables:
```bash
export AZURE_SUBSCRIPTION_ID="..."
export AZURE_RESOURCE_GROUP="..."
```

In [ ]:
# Azure credentials — FILL THESE IN (empty strings pass Git hooks)
AZURE_SUBSCRIPTION_ID = ""
AZURE_RESOURCE_GROUP = ""
AZURE_ML_WORKSPACE = ""
AZURE_REGION = "eastus"

# Validate credentials are set
if not all([AZURE_SUBSCRIPTION_ID, AZURE_RESOURCE_GROUP, AZURE_ML_WORKSPACE]):
    raise ValueError(
        "Azure credentials not set. Fill in AZURE_SUBSCRIPTION_ID, "
        "AZURE_RESOURCE_GROUP, and AZURE_ML_WORKSPACE before running."
    )

print("✅ Azure credentials configured")
print(f"   Subscription: {AZURE_SUBSCRIPTION_ID[:8]}...")
print(f"   Resource Group: {AZURE_RESOURCE_GROUP}")
print(f"   Workspace: {AZURE_ML_WORKSPACE}")
print(f"   Region: {AZURE_REGION}")

## Step 1 — Azure SDK Setup

Install Azure ML SDK and verify authentication.

In [ ]:
import subprocess, sys

# Install Azure ML SDK
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'azure-ai-ml>=1.11.0',
    'azure-identity>=1.15.0',
    'azure-mgmt-compute>=30.0.0',
    'psutil',  # for local memory profiling
], check=True)

print('✅ Azure SDK packages installed')

In [ ]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

# Authenticate using DefaultAzureCredential (uses az login, environment vars, or managed identity)
credential = DefaultAzureCredential()

# Create ML client
ml_client = MLClient(
    credential=credential,
    subscription_id=AZURE_SUBSCRIPTION_ID,
    resource_group_name=AZURE_RESOURCE_GROUP,
    workspace_name=AZURE_ML_WORKSPACE,
)

print(f"✅ Authenticated to Azure ML workspace: {ml_client.workspace_name}")
print(f"   Subscription: {ml_client.subscription_id[:8]}...")

## Step 2 — Azure GPU SKU Comparison

Compare VRAM, cost, and availability of Azure GPU VM SKUs.

**Azure GPU VM families:**
- **NC-series** — NVIDIA Tesla K80, M60 (older, budget GPUs)
- **NCv3-series** — NVIDIA Tesla V100 (16 GB or 32 GB VRAM)
- **ND-series** — NVIDIA Tesla P40 (24 GB VRAM, training-optimized)
- **NDv2-series** — NVIDIA Tesla V100 (32 GB VRAM × 8 GPUs, multi-GPU training)
- **NC A100 v4-series** — NVIDIA A100 (40 GB or 80 GB VRAM, latest generation)

**Key decision:** Match VRAM requirements from Ch.2 formulas to Azure SKU.

In [ ]:
# Azure GPU SKU reference (approximate costs as of 2026, East US region)
# Source: https://azure.microsoft.com/pricing/details/machine-learning/

azure_gpu_skus = {
    # NC-series (older generation, budget)
    "Standard_NC6": {
        "gpu": "Tesla K80",
        "vram_gb": 12,
        "gpus": 1,
        "vcpus": 6,
        "ram_gb": 56,
        "cost_per_hour": 0.90,
        "best_for": "Light inference, learning",
    },
    "Standard_NC12": {
        "gpu": "Tesla K80",
        "vram_gb": 24,
        "gpus": 2,
        "vcpus": 12,
        "ram_gb": 112,
        "cost_per_hour": 1.80,
        "best_for": "Multi-GPU inference",
    },
    # NCv3-series (V100, modern training)
    "Standard_NC6s_v3": {
        "gpu": "Tesla V100",
        "vram_gb": 16,
        "gpus": 1,
        "vcpus": 6,
        "ram_gb": 112,
        "cost_per_hour": 3.06,
        "best_for": "Llama-3-8B inference (FP16)",
    },
    "Standard_NC12s_v3": {
        "gpu": "Tesla V100",
        "vram_gb": 32,
        "gpus": 2,
        "vcpus": 12,
        "ram_gb": 224,
        "cost_per_hour": 6.12,
        "best_for": "Llama-3-8B training (LoRA)",
    },
    # ND-series (P40, training-optimized)
    "Standard_ND6s": {
        "gpu": "Tesla P40",
        "vram_gb": 24,
        "gpus": 1,
        "vcpus": 6,
        "ram_gb": 112,
        "cost_per_hour": 2.07,
        "best_for": "Budget inference (similar to RTX 4090)",
    },
    # NC A100 v4-series (latest, highest performance)
    "Standard_NC24ads_A100_v4": {
        "gpu": "A100 (40GB)",
        "vram_gb": 40,
        "gpus": 1,
        "vcpus": 24,
        "ram_gb": 220,
        "cost_per_hour": 3.67,
        "best_for": "Llama-3-8B training (full fine-tuning)",
    },
    "Standard_NC48ads_A100_v4": {
        "gpu": "A100 (40GB)",
        "vram_gb": 80,  # 2× 40GB GPUs
        "gpus": 2,
        "vcpus": 48,
        "ram_gb": 440,
        "cost_per_hour": 7.35,
        "best_for": "Llama-2-70B inference or multi-GPU training",
    },
    "Standard_NC96ads_A100_v4": {
        "gpu": "A100 (80GB)",
        "vram_gb": 80,
        "gpus": 1,
        "vcpus": 96,
        "ram_gb": 880,
        "cost_per_hour": 10.75,
        "best_for": "Llama-2-70B INT4 inference or large model training",
    },
}

# Display SKU comparison table
import pandas as pd

df_skus = pd.DataFrame.from_dict(azure_gpu_skus, orient='index')
df_skus['cost_per_day'] = df_skus['cost_per_hour'] * 24
df_skus['cost_per_month'] = df_skus['cost_per_day'] * 30

print("Azure GPU SKU Comparison")
print("=" * 80)
print(df_skus[['gpu', 'vram_gb', 'gpus', 'cost_per_hour', 'cost_per_month', 'best_for']].to_string())

## Step 3 — VRAM Estimation for Llama-3-8B on Azure

Use the Ch.2 VRAM formulas to estimate which Azure GPU SKU fits your model.

**Recall from Ch.2:**
```
VRAM_inference = Params + KV Cache + Activations
VRAM_training = Params + Optimizer States + Gradients + Activations
```

We'll calculate for Llama-3-8B at different precisions and batch sizes.

In [ ]:
def vram_budget_inference(
    n_params: int,
    bytes_per_param: float,
    n_layers: int,
    n_heads: int,
    seq_len: int,
    batch_size: int,
    activation_overhead_gb: float = 2.0,
) -> dict:
    """
    Calculate inference VRAM breakdown in GB.
    
    Formula from Ch.2:
        Params = n_params × bytes_per_param
        KV Cache = 2 × n_layers × n_heads × head_dim × seq_len × batch_size × bytes_per_param
        Total = Params + KV Cache + Activations
    """
    params_gb = (n_params * bytes_per_param) / 1e9
    
    # Llama uses GQA (Grouped Query Attention), but for simplicity assume head_dim=128
    head_dim = 128
    kv_cache_gb = (2 * n_layers * n_heads * head_dim * seq_len * batch_size * bytes_per_param) / 1e9
    
    total_gb = params_gb + kv_cache_gb + activation_overhead_gb
    
    # Find matching Azure SKU
    recommended_sku = None
    for sku_name, sku_info in azure_gpu_skus.items():
        if sku_info['vram_gb'] >= total_gb:
            if recommended_sku is None or sku_info['cost_per_hour'] < azure_gpu_skus[recommended_sku]['cost_per_hour']:
                recommended_sku = sku_name
    
    return {
        "params_gb": round(params_gb, 2),
        "kv_cache_gb": round(kv_cache_gb, 2),
        "activations_gb": activation_overhead_gb,
        "total_gb": round(total_gb, 2),
        "recommended_azure_sku": recommended_sku,
        "sku_vram_gb": azure_gpu_skus[recommended_sku]['vram_gb'] if recommended_sku else None,
        "sku_cost_per_hour": azure_gpu_skus[recommended_sku]['cost_per_hour'] if recommended_sku else None,
    }


# Llama-3-8B specs
LLAMA_3_8B_PARAMS = 8_000_000_000
LLAMA_3_8B_LAYERS = 32
LLAMA_3_8B_HEADS = 32

# Test different scenarios
scenarios = [
    ("FP16, batch=1, seq=2048", 2.0, 1, 2048),
    ("FP16, batch=4, seq=2048", 2.0, 4, 2048),
    ("INT8, batch=4, seq=2048", 1.0, 4, 2048),
    ("INT4, batch=8, seq=2048", 0.5, 8, 2048),
]

print("Llama-3-8B VRAM Estimation on Azure")
print("=" * 120)

for name, bytes_per_param, batch_size, seq_len in scenarios:
    result = vram_budget_inference(
        n_params=LLAMA_3_8B_PARAMS,
        bytes_per_param=bytes_per_param,
        n_layers=LLAMA_3_8B_LAYERS,
        n_heads=LLAMA_3_8B_HEADS,
        seq_len=seq_len,
        batch_size=batch_size,
    )
    
    print(f"\n{name}:")
    print(f"  Params:       {result['params_gb']:6.2f} GB")
    print(f"  KV Cache:     {result['kv_cache_gb']:6.2f} GB")
    print(f"  Activations:  {result['activations_gb']:6.2f} GB")
    print(f"  ─────────────────────")
    print(f"  Total VRAM:   {result['total_gb']:6.2f} GB")
    print(f"  Recommended:  {result['recommended_azure_sku']}")
    print(f"  SKU VRAM:     {result['sku_vram_gb']:6.2f} GB")
    print(f"  Cost:         ${result['sku_cost_per_hour']:.2f}/hour (${result['sku_cost_per_hour'] * 24 * 30:.2f}/month)")

## Step 4 — Training VRAM Estimation (Adam Optimizer)

For training, add optimizer states (Adam = 2× params in FP32) and gradients (1× params).

**Formula from Ch.2:**
```
VRAM_training = Params (FP16) + Optimizer States (FP32) + Gradients (FP16) + Activations
                = 2P + 8P + 2P + 8 GB = 12P + 8 GB (for FP16 params, FP32 optimizer)
```

In [ ]:
def vram_budget_training(
    n_params: int,
    param_bytes: float = 2.0,       # FP16 params
    optimizer_bytes: float = 4.0,   # FP32 optimizer states (Adam)
    gradient_bytes: float = 2.0,    # FP16 gradients
    activation_overhead_gb: float = 8.0,  # higher for training
) -> dict:
    """
    Calculate training VRAM breakdown in GB.
    
    Adam optimizer maintains 2 state tensors (momentum + variance) per parameter.
    """
    params_gb = (n_params * param_bytes) / 1e9
    optimizer_states_gb = (n_params * optimizer_bytes * 2) / 1e9  # 2 states (momentum + variance)
    gradients_gb = (n_params * gradient_bytes) / 1e9
    total_gb = params_gb + optimizer_states_gb + gradients_gb + activation_overhead_gb
    
    # Find matching Azure SKU
    recommended_sku = None
    for sku_name, sku_info in azure_gpu_skus.items():
        if sku_info['vram_gb'] >= total_gb:
            if recommended_sku is None or sku_info['cost_per_hour'] < azure_gpu_skus[recommended_sku]['cost_per_hour']:
                recommended_sku = sku_name
    
    return {
        "params_gb": round(params_gb, 2),
        "optimizer_states_gb": round(optimizer_states_gb, 2),
        "gradients_gb": round(gradients_gb, 2),
        "activations_gb": activation_overhead_gb,
        "total_gb": round(total_gb, 2),
        "recommended_azure_sku": recommended_sku,
        "sku_vram_gb": azure_gpu_skus[recommended_sku]['vram_gb'] if recommended_sku else None,
        "sku_cost_per_hour": azure_gpu_skus[recommended_sku]['cost_per_hour'] if recommended_sku else None,
    }


# Llama-3-8B full fine-tuning
result_train = vram_budget_training(n_params=LLAMA_3_8B_PARAMS)

print("Llama-3-8B Training VRAM Estimation (Full Fine-Tuning, Adam)")
print("=" * 80)
print(f"Params (FP16):         {result_train['params_gb']:6.2f} GB")
print(f"Optimizer States:      {result_train['optimizer_states_gb']:6.2f} GB  (FP32 momentum + variance)")
print(f"Gradients (FP16):      {result_train['gradients_gb']:6.2f} GB")
print(f"Activations:           {result_train['activations_gb']:6.2f} GB")
print(f"─────────────────────────────")
print(f"Total VRAM:            {result_train['total_gb']:6.2f} GB")
print(f"Recommended SKU:       {result_train['recommended_azure_sku']}")
print(f"SKU VRAM:              {result_train['sku_vram_gb']:6.2f} GB")
print(f"Cost:                  ${result_train['sku_cost_per_hour']:.2f}/hour")
print(f"                       ${result_train['sku_cost_per_hour'] * 24 * 30:.2f}/month (24/7 usage)")
print()
print("💡 Tip: Use LoRA fine-tuning to reduce training VRAM by 10×")
print("   LoRA trains only adapter weights (~2 GB) → fits on Standard_NC6s_v3 (16 GB)")

## Step 5 — Memory Profiling on Azure ML Compute Cluster (Dry Run)

Demonstrate how to profile actual VRAM usage on an Azure ML compute cluster.

**Note:** This is a **local simulation** since spinning up Azure GPU VMs for every learner is expensive.  
In production, you would:
1. Create an Azure ML compute cluster with the target GPU SKU
2. Submit a profiling job that loads your model and logs VRAM usage
3. Use `torch.cuda.memory_allocated()` to track actual memory consumption

**For learning purposes**, we'll simulate the profiling output here.

In [ ]:
# Simulate Azure ML compute cluster memory profiling
# (In production, this would run on an actual Azure GPU VM)

import json

def simulate_azure_memory_profile(sku_name: str, model_name: str, precision: str, batch_size: int):
    """
    Simulate what an Azure ML profiling job would return.
    
    In production, this would:
    1. Spin up Azure ML compute with specified SKU
    2. Load model onto GPU
    3. Run inference with specified batch size
    4. Log torch.cuda.memory_allocated() and torch.cuda.max_memory_allocated()
    5. Return profiling metrics
    """
    sku_info = azure_gpu_skus.get(sku_name, {})
    
    # Simulate realistic VRAM usage (based on Ch.2 formulas)
    if model_name == "Llama-3-8B" and precision == "FP16":
        baseline_vram = 16.0  # params
        kv_cache_per_batch = 4.0
        activations = 2.0
        total_vram = baseline_vram + (kv_cache_per_batch * batch_size) + activations
    elif model_name == "Llama-3-8B" and precision == "INT4":
        baseline_vram = 4.0
        kv_cache_per_batch = 1.0
        activations = 2.0
        total_vram = baseline_vram + (kv_cache_per_batch * batch_size) + activations
    else:
        total_vram = 0.0
    
    max_vram = sku_info.get('vram_gb', 0)
    utilization_pct = (total_vram / max_vram * 100) if max_vram > 0 else 0
    
    return {
        "sku": sku_name,
        "gpu": sku_info.get('gpu', 'Unknown'),
        "max_vram_gb": max_vram,
        "model": model_name,
        "precision": precision,
        "batch_size": batch_size,
        "allocated_vram_gb": round(total_vram, 2),
        "utilization_pct": round(utilization_pct, 1),
        "fits": total_vram <= max_vram,
        "headroom_gb": round(max_vram - total_vram, 2) if total_vram <= max_vram else 0,
    }


# Simulate profiling for different Azure SKUs
test_configs = [
    ("Standard_NC6s_v3", "Llama-3-8B", "FP16", 1),
    ("Standard_NC6s_v3", "Llama-3-8B", "FP16", 4),
    ("Standard_ND6s", "Llama-3-8B", "FP16", 1),
    ("Standard_NC24ads_A100_v4", "Llama-3-8B", "INT4", 8),
]

print("Azure ML Memory Profiling Simulation")
print("=" * 100)

for sku, model, precision, batch in test_configs:
    profile = simulate_azure_memory_profile(sku, model, precision, batch)
    
    status = "✅ FITS" if profile['fits'] else "❌ OOM"
    print(f"\n{sku} ({profile['gpu']}, {profile['max_vram_gb']} GB)")
    print(f"  Model: {model} @ {precision}, batch={batch}")
    print(f"  Allocated: {profile['allocated_vram_gb']} GB ({profile['utilization_pct']}% utilization)")
    print(f"  Headroom:  {profile['headroom_gb']} GB")
    print(f"  Status:    {status}")

## Step 6 — Batch Size vs. VRAM Tradeoff (Azure Cost Optimization)

Visualize how batch size affects VRAM and throughput on Azure.

**Key insight from Ch.2:** Throughput scales with batch size, but so does VRAM consumption (via KV cache).  
Larger Azure GPU SKUs enable higher batch sizes → better throughput → lower cost per request.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Calculate VRAM and cost for different batch sizes on Azure SKUs
def analyze_batch_scaling(sku_name: str, model_params: int, precision_bytes: float):
    sku = azure_gpu_skus[sku_name]
    max_vram = sku['vram_gb']
    
    batch_sizes = []
    vram_usage = []
    throughput_estimate = []
    cost_per_1k_requests = []
    
    # Baseline VRAM (params + activations)
    params_gb = (model_params * precision_bytes) / 1e9
    activations_gb = 2.0
    baseline_vram = params_gb + activations_gb
    
    # KV cache per batch (simplified: ~4 GB per batch for Llama-3-8B FP16, seq=2048)
    kv_per_batch = 4.0 if precision_bytes == 2.0 else 1.0  # INT4 uses less
    
    for batch in range(1, 17):
        total_vram = baseline_vram + (kv_per_batch * batch)
        
        if total_vram > max_vram:
            break  # OOM
        
        batch_sizes.append(batch)
        vram_usage.append(total_vram)
        
        # Throughput estimate: batch × 40 tokens/sec (Llama-3-8B baseline)
        throughput = batch * 40
        throughput_estimate.append(throughput)
        
        # Cost per 1k requests (assuming 2048 tokens per request = ~51 seconds at 40 tok/s)
        seconds_per_request = 2048 / 40 / batch  # parallel batching reduces time
        hours_per_1k_requests = (1000 * seconds_per_request) / 3600
        cost_per_1k = hours_per_1k_requests * sku['cost_per_hour']
        cost_per_1k_requests.append(cost_per_1k)
    
    return batch_sizes, vram_usage, throughput_estimate, cost_per_1k_requests


# Analyze two Azure SKUs
sku1 = "Standard_NC6s_v3"  # 16 GB V100
sku2 = "Standard_NC24ads_A100_v4"  # 40 GB A100

batch1, vram1, throughput1, cost1 = analyze_batch_scaling(sku1, LLAMA_3_8B_PARAMS, 2.0)
batch2, vram2, throughput2, cost2 = analyze_batch_scaling(sku2, LLAMA_3_8B_PARAMS, 2.0)

# Plot
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# VRAM usage
axes[0].plot(batch1, vram1, 'o-', label=f'{sku1} ({azure_gpu_skus[sku1]["vram_gb"]} GB)', linewidth=2)
axes[0].plot(batch2, vram2, 's-', label=f'{sku2} ({azure_gpu_skus[sku2]["vram_gb"]} GB)', linewidth=2)
axes[0].axhline(y=azure_gpu_skus[sku1]['vram_gb'], color='blue', linestyle='--', alpha=0.5, label='V100 VRAM limit')
axes[0].axhline(y=azure_gpu_skus[sku2]['vram_gb'], color='orange', linestyle='--', alpha=0.5, label='A100 VRAM limit')
axes[0].set_xlabel('Batch Size', fontsize=12)
axes[0].set_ylabel('VRAM Usage (GB)', fontsize=12)
axes[0].set_title('VRAM vs. Batch Size', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Throughput
axes[1].plot(batch1, throughput1, 'o-', label=sku1, linewidth=2)
axes[1].plot(batch2, throughput2, 's-', label=sku2, linewidth=2)
axes[1].set_xlabel('Batch Size', fontsize=12)
axes[1].set_ylabel('Throughput (tokens/sec)', fontsize=12)
axes[1].set_title('Throughput vs. Batch Size', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Cost per 1k requests
axes[2].plot(batch1, cost1, 'o-', label=sku1, linewidth=2)
axes[2].plot(batch2, cost2, 's-', label=sku2, linewidth=2)
axes[2].set_xlabel('Batch Size', fontsize=12)
axes[2].set_ylabel('Cost per 1k Requests ($)', fontsize=12)
axes[2].set_title('Cost Efficiency vs. Batch Size', fontsize=14, fontweight='bold')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('img/azure_batch_scaling_tradeoff.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n💡 Key Insights:")
print(f"   • {sku1} (16 GB): Max batch = {max(batch1)}, throughput = {max(throughput1)} tok/s, cost = ${min(cost1):.2f}/1k req")
print(f"   • {sku2} (40 GB): Max batch = {max(batch2)}, throughput = {max(throughput2)} tok/s, cost = ${min(cost2):.2f}/1k req")
print(f"   • A100 enables {max(batch2) / max(batch1):.1f}× higher batch size → {max(throughput2) / max(throughput1):.1f}× throughput")
print(f"   • Cost per request drops {cost1[0] / min(cost2):.1f}× with optimal batching on A100")

## Step 7 — Choosing the Right Azure GPU (Decision Tree)

**Decision tree for Azure GPU selection:**

```
┌─────────────────────────────────────────────────────────────┐
│ What are you doing?                                         │
└─────────────────────────────────────────────────────────────┘
    │
    ├─ Inference (Llama-3-8B, FP16, batch=1)
    │  → Standard_NC6s_v3 (V100 16GB, $3.06/hr)
    │  → Fits: 22 GB < 24 GB ✅, but batch=1 only
    │
    ├─ Inference (Llama-3-8B, FP16, batch=4 for throughput)
    │  → Standard_NC24ads_A100_v4 (A100 40GB, $3.67/hr)
    │  → Fits: 28 GB < 40 GB ✅, batch=4 → 4× throughput
    │
    ├─ Inference (Llama-3-8B, INT4, batch=8)
    │  → Standard_ND6s (P40 24GB, $2.07/hr)
    │  → Fits: 14 GB < 24 GB ✅, INT4 saves $1/hr vs V100
    │
    ├─ Training (Llama-3-8B, LoRA fine-tuning)
    │  → Standard_NC6s_v3 (V100 16GB, $3.06/hr)
    │  → LoRA adapters = 2 GB → total 18 GB ✅
    │
    ├─ Training (Llama-3-8B, full fine-tuning, Adam)
    │  → Standard_NC96ads_A100_v4 (A100 80GB, $10.75/hr)
    │  → Full training = 104 GB → need A100 80GB ✅
    │
    └─ Inference (Llama-2-70B, INT4)
       → Standard_NC96ads_A100_v4 (A100 80GB, $10.75/hr)
       → 70B INT4 = 35 GB params + 21 GB KV = 56 GB ✅
```

In [ ]:
# Azure GPU Decision Helper
def recommend_azure_sku(
    model_size_b: float,
    precision: str,
    task: str,  # "inference" or "training"
    batch_size: int = 1,
    seq_len: int = 2048,
):
    """
    Recommend the cheapest Azure GPU SKU that fits your requirements.
    """
    # Convert precision to bytes
    precision_map = {"FP32": 4.0, "FP16": 2.0, "BF16": 2.0, "INT8": 1.0, "INT4": 0.5}
    bytes_per_param = precision_map.get(precision, 2.0)
    
    # Estimate VRAM
    if task == "inference":
        params_gb = model_size_b * bytes_per_param
        kv_cache_gb = 0.5 * batch_size * seq_len / 512  # simplified estimate
        activations_gb = 2.0
        total_vram = params_gb + kv_cache_gb + activations_gb
    elif task == "training":
        params_gb = model_size_b * 2.0  # FP16 params
        optimizer_gb = model_size_b * 4.0 * 2  # FP32 optimizer states
        gradients_gb = model_size_b * 2.0
        activations_gb = 8.0
        total_vram = params_gb + optimizer_gb + gradients_gb + activations_gb
    else:
        raise ValueError("task must be 'inference' or 'training'")
    
    # Find cheapest SKU that fits
    candidates = []
    for sku_name, sku_info in azure_gpu_skus.items():
        if sku_info['vram_gb'] >= total_vram:
            candidates.append((sku_name, sku_info['cost_per_hour'], sku_info['vram_gb']))
    
    if not candidates:
        return {
            "error": f"No Azure SKU found with {total_vram:.1f} GB VRAM",
            "required_vram_gb": round(total_vram, 2),
        }
    
    # Sort by cost (ascending)
    candidates.sort(key=lambda x: x[1])
    best_sku_name, best_cost, best_vram = candidates[0]
    
    return {
        "model_size_b": model_size_b,
        "precision": precision,
        "task": task,
        "batch_size": batch_size,
        "required_vram_gb": round(total_vram, 2),
        "recommended_sku": best_sku_name,
        "sku_vram_gb": best_vram,
        "sku_cost_per_hour": best_cost,
        "sku_cost_per_month": round(best_cost * 24 * 30, 2),
        "headroom_gb": round(best_vram - total_vram, 2),
    }


# Test different scenarios
test_scenarios = [
    (8.0, "FP16", "inference", 1, 2048),
    (8.0, "FP16", "inference", 4, 2048),
    (8.0, "INT4", "inference", 8, 2048),
    (8.0, "FP16", "training", 1, 2048),
    (70.0, "INT4", "inference", 2, 4096),
]

print("Azure GPU Recommendation Engine")
print("=" * 120)

for model_size, precision, task, batch, seq_len in test_scenarios:
    result = recommend_azure_sku(model_size, precision, task, batch, seq_len)
    
    if "error" in result:
        print(f"\n❌ {result['error']}")
        continue
    
    print(f"\n📊 Model: {result['model_size_b']}B params @ {precision}, {task}, batch={batch}")
    print(f"   Required VRAM: {result['required_vram_gb']} GB")
    print(f"   Recommended:   {result['recommended_sku']}")
    print(f"   SKU VRAM:      {result['sku_vram_gb']} GB ({result['headroom_gb']} GB headroom)")
    print(f"   Cost:          ${result['sku_cost_per_hour']:.2f}/hour (${result['sku_cost_per_month']:.2f}/month)")

## Step 8 — Production Checklist: Deploying on Azure ML

**Before deploying your model to Azure ML compute:**

1. ✅ **Calculate exact VRAM requirements** (use formulas from this notebook)
2. ✅ **Choose Azure GPU SKU** with 10-20% headroom (not 100% utilization)
3. ✅ **Request GPU quota** in Azure subscription (NC/ND series often have low default quotas)
4. ✅ **Test with smaller batch size first** (batch=1) before scaling up
5. ✅ **Monitor GPU memory** during inference using `torch.cuda.memory_allocated()`
6. ✅ **Set up auto-scaling** in Azure ML to handle variable load
7. ✅ **Use INT8/INT4 quantization** (Ch.3) to reduce costs by 2-4×

**Common mistakes:**
- ❌ Forgetting KV cache scaling (batch × seq_len)
- ❌ Running at 95%+ VRAM utilization (causes fragmentation OOM)
- ❌ Not checking Azure GPU quota before deploying
- ❌ Choosing cheapest SKU without headroom (false economy — OOM kills throughput)

**Next steps:**
- Ch.3: Quantization (reduce VRAM by 4× with INT4)
- Ch.4: Parallelism (split across multiple Azure GPUs with DeepSpeed)
- Ch.5: Inference Optimization (deploy vLLM on Azure Kubernetes Service)

## Summary — What You Learned

You now know how to:

1. ✅ **Map VRAM requirements to Azure GPU SKUs** (NC, ND, NC A100 series)
2. ✅ **Estimate inference and training memory** for any model size and precision
3. ✅ **Optimize batch size** to balance throughput and cost on Azure
4. ✅ **Choose the cheapest Azure GPU** that fits your model (with headroom)
5. ✅ **Avoid OOM errors** by calculating exact VRAM before deploying

**Key formulas from Ch.2 (applied to Azure):**

| Metric | Formula |
|--------|---------|
| Inference VRAM | `Params + KV Cache + Activations` |
| Training VRAM | `Params + Optimizer States (2× FP32) + Gradients + Activations` |
| KV Cache | `2 × layers × heads × head_dim × seq_len × batch_size × bytes` |
| Azure cost/month | `sku_hourly_rate × 24 × 30` |

**Critical insight:** A larger Azure GPU (A100 vs V100) costs 20% more per hour, but enables 4× higher batch size → 3× lower cost per request.

**Bridge to Ch.3:** Quantization reduces Azure costs by 4× (INT4) while maintaining quality. The same VRAM formulas apply — just change `bytes_per_param` from 2.0 to 0.5.